# Bab 1 — Introduction to Clustering Algorithms

Notebook pendamping Bab 1 Level 4.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

- Menjelaskan apa itu algoritma clustering dan mengapa ada banyak jenisnya.
- Membedakan *hard clustering* dan *soft clustering*.
- Memakai pola `fit_predict` dan `labels_` pada lima algoritma scikit-learn.
- Membaca perbedaan hasil lima algoritma pada dua dataset.

## Konsep singkat

Satu tujuan (mengelompokkan data tanpa label) bisa dikerjakan dengan banyak cara karena definisi "kelompok" berbeda-beda: dekat dengan pusat yang sama (K-Means), bergabung bertahap (hierarchical), daerah padat (DBSCAN), distribusi probabilitas (GMM), atau puncak kepadatan (Mean Shift).

## Dataset

Dua dataset sintetis, masing-masing 300 titik dan dua fitur, dengan `random_state=42`: `make_blobs` (tiga gerombol bulat) dan `make_moons` (dua bulan sabit). Label asli dibuang karena clustering bersifat *unsupervised*.

## Langkah 0: Pindah ke folder utama proyek

Sel ini membuat jalur relatif berfungsi bila notebook dibuka dari folder `notebooks/level-4/`.

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Langkah 1: Siapkan alat

Kita memanggil NumPy, pandas, Matplotlib, dan lima algoritma. `GaussianMixture` berada di `sklearn.mixture`, bukan `sklearn.cluster`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
    MeanShift,
    estimate_bandwidth,
)
from sklearn.mixture import GaussianMixture

## Langkah 2: Buat dua dataset

`X_blobs, _ = ...` menyimpan data ke `X_blobs` dan membuang label asli ke `_`.

In [ ]:
# Dataset 1: tiga gerombol bulat yang terpisah jelas
X_blobs, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Dataset 2: dua bentuk "bulan sabit" yang saling mengait
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)

print("Bentuk X_blobs:", X_blobs.shape)
print("Bentuk X_moons:", X_moons.shape)
print("\nLima baris pertama X_blobs:")
print(pd.DataFrame(X_blobs, columns=["x1", "x2"]).head().round(3))

## Langkah 3: Lihat kedua dataset

Selalu lihat data mentah sebelum memilih algoritma.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, X, title in [
    (axes[0], X_blobs, "Dataset 1: blobs (gerombol bulat)"),
    (axes[1], X_moons, "Dataset 2: moons (bulan sabit)"),
]:
    ax.scatter(X[:, 0], X[:, 1], s=18, color="#444444")
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
plt.tight_layout()
plt.show()

## Langkah 4: Buat lima algoritma lewat satu fungsi

Kelimanya dibuat dengan cara yang sama: `NamaAlgoritma(pengaturan)`. DBSCAN dan Mean Shift tidak meminta jumlah kelompok.

In [ ]:
def build_models(n_clusters, eps, bandwidth):
    """Membuat lima algoritma dengan pengaturan awal yang masuk akal."""
    return {
        "K-Means": KMeans(n_clusters=n_clusters, n_init=10, random_state=42),
        "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=n_clusters, linkage="ward"),
        "DBSCAN": DBSCAN(eps=eps, min_samples=5),
        "GMM": GaussianMixture(n_components=n_clusters, random_state=42),
        "Mean Shift": MeanShift(bandwidth=bandwidth),
    }

## Langkah 5: Jalankan semua algoritma pada data blobs

`fit_predict` mengembalikan satu label per titik. Perhatikan label `-1` (noise) pada DBSCAN dan jumlah kelompok Mean Shift yang ditentukan sendiri oleh algoritma.

In [ ]:
bw_blobs = estimate_bandwidth(X_blobs, quantile=0.2, random_state=42)
models = build_models(n_clusters=3, eps=0.7, bandwidth=bw_blobs)

print(f"Bandwidth Mean Shift (hasil estimate_bandwidth): {bw_blobs:.3f}\n")
for name, model in models.items():
    labels = model.fit_predict(X_blobs)
    values, counts = np.unique(labels, return_counts=True)
    print(f"{name:<20} label unik: {values.tolist()}  jumlah anggota: {counts.tolist()}")

## Langkah 6: Bandingkan kemampuan tiap objek

`hasattr` memeriksa apakah sebuah objek punya anggota tertentu, tanpa memanggilnya.

In [ ]:
rows = []
for name, model in models.items():
    rows.append(
        {
            "Algoritma": name,
            "punya labels_": hasattr(model, "labels_"),
            "punya predict()": hasattr(model, "predict"),
            "punya predict_proba()": hasattr(model, "predict_proba"),
            "punya pusat": hasattr(model, "cluster_centers_") or hasattr(model, "means_"),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

## Langkah 7: Jalankan lima algoritma pada dua dataset

Pengaturan `eps` dan jumlah kelompok dipilih setelah melihat gambar data. Pada data nyata, menilai pengaturan yang baik adalah topik Level 5.

In [ ]:
bw_moons = estimate_bandwidth(X_moons, quantile=0.2, random_state=42)
settings = {
    "blobs": dict(X=X_blobs, n_clusters=3, eps=0.7, bandwidth=bw_blobs),
    "moons": dict(X=X_moons, n_clusters=2, eps=0.2, bandwidth=bw_moons),
}

results = {}  # (nama dataset, nama algoritma) -> label
for data_name, cfg in settings.items():
    X = cfg["X"]
    models = build_models(cfg["n_clusters"], cfg["eps"], cfg["bandwidth"])
    for algo_name, model in models.items():
        results[(data_name, algo_name)] = model.fit_predict(X)

print("Jumlah kelompok (tanpa noise) dan jumlah noise per algoritma:")
for (data_name, algo_name), labels in results.items():
    n_clusters = len(set(labels) - {-1})
    n_noise = int(np.sum(labels == -1))
    print(f"{data_name:<6} {algo_name:<20} kelompok: {n_clusters}  noise: {n_noise}")

## Langkah 8: Gambar hasilnya

Baris atas: blobs. Baris bawah: moons. Tanda x abu-abu adalah noise DBSCAN.

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9", "#F0E442"]
algo_names = list(build_models(3, 0.7, 1.0).keys())

fig, axes = plt.subplots(2, 5, figsize=(15, 6.2))
for row, (data_name, cfg) in enumerate(settings.items()):
    X = cfg["X"]
    for col, algo_name in enumerate(algo_names):
        ax = axes[row, col]
        labels = results[(data_name, algo_name)]
        for k in sorted(set(labels)):
            mask = labels == k
            if k == -1:  # noise DBSCAN digambar abu-abu
                ax.scatter(X[mask, 0], X[mask, 1], s=16, color="#999999", marker="x")
            else:
                ax.scatter(X[mask, 0], X[mask, 1], s=16, color=palette[k % len(palette)])
        if row == 0:
            ax.set_title(algo_name)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[row, 0].set_ylabel(data_name)
plt.tight_layout()
plt.show()

## Interpretasi

- Pada **blobs**, K-Means, Ward, dan GMM memberi tiga kelompok yang sama ukurannya. DBSCAN menandai beberapa titik tepi sebagai noise. Mean Shift menemukan satu kelompok kecil tambahan.
- Pada **moons**, K-Means, Ward, dan GMM memotong kedua bulan sabit. DBSCAN mengikuti bentuknya.
- Hanya GMM yang punya `predict_proba` (soft clustering). DBSCAN dan Hierarchical tidak punya `predict`.
- Tidak ada algoritma yang unggul di kedua dataset.

## Latihan

1. Ubah `n_clusters` K-Means pada blobs menjadi 4. Apa yang terjadi pada salah satu gerombol?
2. Ubah `eps` DBSCAN pada blobs menjadi 1.0 dan 0.4. Berapa jumlah kelompok dan noise?
3. Ubah `eps` DBSCAN pada moons menjadi 0.05 dan 0.5.
4. Ubah `quantile` pada `estimate_bandwidth` menjadi 0.1, 0.3, dan 0.5. Berapa jumlah kelompok Mean Shift?
5. Latih `GaussianMixture(n_components=2, random_state=42)` pada `X_moons` dan hitung berapa titik yang `predict_proba` tertingginya di bawah 0,9.

Jawaban ada di halaman bab.

## Ringkasan

- Ada banyak algoritma clustering karena ada banyak definisi "kelompok".
- Lima pendekatan: partition-based, hierarchical, density-based, model-based, mode-seeking.
- Cara pakai di scikit-learn mirip: buat objek, `fit_predict`, baca label.
- Tidak ada algoritma terbaik untuk semua data. Bab 7 menyusun panduan memilih.